# 01 · Read flux CSVs and PWB tlag summaries → Parquet

First processing step. Each EddyPro flux output (FLUXNET CSV format, one file
per [processing version](../docs/processing-versions.md)) is read with `diive`
and written back out as a Parquet file for fast, type-safe reloading in the
downstream analysis.

- **Input:** `data/00-eddypro_fluxes_level-1/` — the raw EddyPro FLUXNET CSVs.
- **Output:** `data/01-eddypro_fluxes_level-1_parquet/` — one `.parquet` per version.

The reader is the same one used in diive's example data
(`load_exampledata_EDDYPRO_FLUXNET_CSV_30MIN`): `ReadFileType` with
`filetype='EDDYPRO-FLUXNET-CSV-30MIN'`. Saving uses `diive.core.io.files.save_parquet`.

The notebook also converts the PWB time-lag summaries for the `*-4` variants
(see the last section), which contain time-lag results rather than fluxes.

## Imports

In [15]:
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

from diive.core.io.filereader import ReadFileType
from diive.core.io.files import save_parquet

NB_START = datetime.now()  # notebook start time (reported in the last cell)

## Configuration

In [16]:
# Input / output folders (relative to the notebooks/ directory).
INDIR = Path("../data/00-eddypro_fluxes_level-1")
OUTDIR = Path("../data/01-eddypro_fluxes_level-1_parquet")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Discover the input CSVs.
csv_files = sorted(INDIR.glob("eddypro_*_adv.csv"))
print(f"Found {len(csv_files)} flux file(s):")
for f in csv_files:
    print(f"  {f.name}")

Found 6 flux file(s):
  eddypro_LGR-1_FR-20260521-103649_fluxnet_2026-05-21T234627_adv.csv
  eddypro_LGR-2R_FR-20260527-195011_fluxnet_2026-05-28T092652_adv.csv
  eddypro_LGR-3_2021_2_2022_1_LGR_eddypro_CH-CHA_FR-20240809-181345_fluxnet_2024-08-10T185331_adv.csv
  eddypro_QCL-1_FR-20260521-103816_fluxnet_2026-05-22T021605_adv.csv
  eddypro_QCL-2R_FR-20260527-194920_fluxnet_2026-05-28T112056_adv.csv
  eddypro_QCL-3_2020_4_5_2021_1_QCL_eddypro_CH-CHA_FR-20240809-181351_fluxnet_2024-08-10T220247_adv.csv


## Version codes

In [17]:
def version_code(filename: str) -> str:
    """Extract the processing-version code (e.g. 'LGR-1', 'QCL-2R') from a filename."""
    match = re.search(r"eddypro_([A-Z]+-\d+R?)_", filename)
    if not match:
        raise ValueError(f"Could not extract a version code from: {filename}")
    return match.group(1)


# Map each input file to its version code (the output Parquet name).
for f in csv_files:
    print(f"{version_code(f.name):>8}  <-  {f.name}")

   LGR-1  <-  eddypro_LGR-1_FR-20260521-103649_fluxnet_2026-05-21T234627_adv.csv
  LGR-2R  <-  eddypro_LGR-2R_FR-20260527-195011_fluxnet_2026-05-28T092652_adv.csv
   LGR-3  <-  eddypro_LGR-3_2021_2_2022_1_LGR_eddypro_CH-CHA_FR-20240809-181345_fluxnet_2024-08-10T185331_adv.csv
   QCL-1  <-  eddypro_QCL-1_FR-20260521-103816_fluxnet_2026-05-22T021605_adv.csv
  QCL-2R  <-  eddypro_QCL-2R_FR-20260527-194920_fluxnet_2026-05-28T112056_adv.csv
   QCL-3  <-  eddypro_QCL-3_2020_4_5_2021_1_QCL_eddypro_CH-CHA_FR-20240809-181351_fluxnet_2024-08-10T220247_adv.csv


## Read each CSV and save as Parquet

`ReadFileType` returns the data (`data_df`) and a metadata table (`metadata_df`,
the variable units). The data frame is written to Parquet named after the
version code, so downstream code can load e.g. `LGR-1.parquet` directly.

In [18]:
saved = {}
for f in csv_files:
    code = version_code(f.name)
    print(f"\n=== {code} ===")

    loaddatafile = ReadFileType(
        filetype="EDDYPRO-FLUXNET-CSV-30MIN",
        filepath=str(f),
        data_nrows=None,
        output_middle_timestamp=True,
    )
    data_df, metadata_df = loaddatafile.get_filedata()
    print(f"  read {data_df.shape[0]} rows x {data_df.shape[1]} cols")

    filepath = save_parquet(filename=code, data=data_df, outpath=str(OUTDIR))
    saved[code] = filepath


=== LGR-1 ===


> Reading file eddypro_LGR-1_FR-20260521-103649_fluxnet_2026-05-21T234627_adv.csv ...

F:\dev\diive\diive\core\io\filereader.py:587: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[_temp_parsed_index_col] = ''


  read 7805 rows x 484 cols


> Saved file ..\data\01-eddypro_fluxes_level-1_parquet\LGR-1.parquet (0.188 seconds).

...


=== QCL-3 ===


> Reading file 
eddypro_QCL-3_2020_4_5_2021_1_QCL_eddypro_CH-CHA_FR-20240809-181351_fluxnet_2024-08-10T220247_adv.csv ...

F:\dev\diive\diive\core\io\filereader.py:587: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[_temp_parsed_index_col] = ''


  read 20790 rows x 482 cols


> Saved file ..\data\01-eddypro_fluxes_level-1_parquet\QCL-3.parquet (0.404 seconds).

## Verify

Reload one Parquet file to confirm the round-trip and inspect a few of the
study-relevant columns (fluxes + time-lag diagnostics).

In [19]:
from diive.core.io.files import load_parquet

print("Saved Parquet files:")
for code, path in saved.items():
    print(f"  {code}: {path}")

check = load_parquet(filepath=saved["LGR-1"])
cols = [c for c in ["FN2O", "FCH4", "N2O_TLAG_USED", "CH4_TLAG_USED"] if c in check.columns]
print(f"\nLGR-1: {check.shape[0]} rows x {check.shape[1]} cols")
check[cols].describe()

Saved Parquet files:
  LGR-1: ..\data\01-eddypro_fluxes_level-1_parquet\LGR-1.parquet
  LGR-2R: ..\data\01-eddypro_fluxes_level-1_parquet\LGR-2R.parquet
  LGR-3: ..\data\01-eddypro_fluxes_level-1_parquet\LGR-3.parquet
  QCL-1: ..\data\01-eddypro_fluxes_level-1_parquet\QCL-1.parquet
  QCL-2R: ..\data\01-eddypro_fluxes_level-1_parquet\QCL-2R.parquet
  QCL-3: ..\data\01-eddypro_fluxes_level-1_parquet\QCL-3.parquet


> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\LGR-1.parquet (0.044 seconds).


LGR-1: 7805 rows x 484 cols


,FN2O,FCH4,N2O_TLAG_USED,CH4_TLAG_USED
count,7731.000000,7731.000000,7731.000000,7731.000000
mean,1.821938,15.252026,3.369681,4.024648
std,9.197670,221.503720,3.196333,3.900663
min,-374.735000,-7194.450000,-1.000000,-1.000000
25%,0.103841,-11.579100,1.600000,1.300000
50%,0.723384,4.498810,1.850000,2.250000
75%,2.550460,27.070200,5.500000,8.200000
max,205.037000,5268.080000,10.000000,10.000000


## PWB tlag summaries

The `*-4` PWB variants (`LGR-4`, `QCL-4`) produce per-chunk time-lag results, not
fluxes (see [processing steps](../docs/processing-steps.md)). Their CSVs in
`data/00-pwb_tlag_summary/` are plain tables (one row per processed chunk), so
they are read with pandas rather than `ReadFileType`.

Only chunks whose start is on the 30-minute wall-clock grid are kept. The leading
partial chunks (off-grid starts like 10:10) are dropped so the series aligns to
the same regular timestamp grid as the flux data; otherwise `load_parquet` would
reindex them onto the grid and null their values.

The result is written to a separate folder
(`data/01-pwb_tlag_summary_parquet/`) with a `pwb_tlag` tag in the filename
(e.g. `LGR-4_pwb_tlag.parquet`), so it stays distinct from the flux Parquets and
is not picked up by the column-subsetting in notebook 02.

In [20]:
PWB_INDIR = Path("../data/00-pwb_tlag_summary")
PWB_OUTDIR = Path("../data/01-pwb_tlag_summary_parquet")
PWB_OUTDIR.mkdir(parents=True, exist_ok=True)

pwb_files = sorted(PWB_INDIR.glob("*_detect_and_remove_tlag_summary.csv"))
print(f"Found {len(pwb_files)} PWB summary file(s):")

pwb_saved = {}
for f in pwb_files:
    code = re.match(r"([A-Z]+-\d+)_", f.name).group(1)  # e.g. 'LGR-4'
    df = pd.read_csv(f, parse_dates=["timestamp"]).set_index("timestamp")

    # Keep only chunks whose start is on the 30-min wall-clock grid. The leading
    # partial chunks (e.g. 10:10) are off-grid and would not align to the regular
    # timestamp grid used by the flux data, so they are dropped.
    on_grid = df.index.minute.isin([0, 30]) & (df.index.second == 0)
    n_off = int((~on_grid).sum())
    df = df[on_grid]

    # The PWB timestamp is the chunk START time; name the index so diive's
    # load_parquet accepts it.
    df.index.name = "TIMESTAMP_START"
    print(f"  {code}: {df.shape[0]} rows x {df.shape[1]} cols ({n_off} off-grid chunks dropped)")
    filepath = save_parquet(filename=f"{code}_pwb_tlag", data=df, outpath=str(PWB_OUTDIR))
    pwb_saved[code] = filepath

Found 2 PWB summary file(s):
  LGR-4: 7805 rows x 44 cols (15 off-grid chunks dropped)


> Saved file ..\data\01-pwb_tlag_summary_parquet\LGR-4_pwb_tlag.parquet (0.080 seconds).

  QCL-4: 9520 rows x 44 cols (8 off-grid chunks dropped)


> Saved file ..\data\01-pwb_tlag_summary_parquet\QCL-4_pwb_tlag.parquet (0.055 seconds).

## Runtime

In [21]:
NB_END = datetime.now()
print(f"Start:    {NB_START:%Y-%m-%d %H:%M:%S}")
print(f"End:      {NB_END:%Y-%m-%d %H:%M:%S}")
print(f"Runtime:  {NB_END - NB_START}")

Start:    2026-06-08 16:33:35
End:      2026-06-08 16:33:58
Runtime:  0:00:22.538758
